# Inspect an observed schema

**Goal:** explore types, missing fields and optional value retention in nested data.

No database is needed. Run cells in order with the BONDI Python environment.
The parser accepts dictionaries from MongoDB, Couchbase or files and describes
only the data supplied, not rules for every possible document.

Companion guide: [schema inspection](../docs/schemas.md).

In [ ]:
from pprint import pprint

from bondi.schema import ARRAY_ELEMENT, Schema

documents = [
    {"city": "Sydney", "rating": 4, "items": [{"price": 10}, {"price": 20}]},
    {"city": "Sydney", "rating": None, "items": [{"price": 10}, {}]},
    {"city": "Melbourne"},
]
schema = Schema.from_documents(documents, values="full")
print(schema)
print("Top-level:", list(schema.fields))
print("All paths:", list(schema.flat_fields))

## Counts, types and missingness

Null means the key was present with value `None`. Missing means its parent object
had no such key. Array-element counts can exceed the number of root documents.


In [ ]:
for path, field in schema.flat_fields.items():
    print(
        path,
        "count=",
        field.count,
        "parents=",
        field.parent_count,
        "missing=",
        field.missing_count,
        "types=",
        list(field.types),
    )

price = schema["items.[].price"]
assert (price.count, price.parent_count, price.missing_count) == (3, 4, 1)
assert schema["rating"].types["null"].count == 1
assert schema["rating"].missing_count == 1
assert schema["items.[]"].missing_count is None
assert schema["items"].types["array"].elements.types["object"].fields["price"] is price

## Choose what values to retain

`none` is the default. `distinct` retains unique values; `full` also retains exact
frequencies. Both enabled modes grow with distinct-value cardinality. Counts of
fields and types are always available, even when values are not retained.


In [ ]:
for mode in ("none", "distinct", "full"):
    observed = Schema.from_documents(documents, values=mode)
    city = observed["city"].types["str"]
    print(mode, "count=", city.count, "values=", city.values, "counts=", city.value_counts)

assert schema["city"].types["str"].value_counts == {"Sydney": 2, "Melbourne": 1}
assert price.types["int"].value_counts == {10: 2, 20: 1}

## Mixed types remain useful for workload generation

A field may contain a primitive sometimes and an object at other times. It still
belongs in `primitive_fields`. Values stay separated by type, including bool/int/float.


In [ ]:
mixed = Schema.from_documents(
    [
        {"a": True},
        {"a": 1},
        {"a": 1.0},
        {"a": {"b": "nested"}},
        {"a": [2]},
    ],
    values="full",
)
for path, field in mixed.primitive_fields.items():
    for name, variant in field.types.items():
        if variant.values is not None:
            print(path, name, variant.value_counts)
assert "a" in mixed.primitive_fields
assert len(mixed["a"].types) == 5

## Add a batch

Field references remain live. The retention policy stays fixed. A field introduced
in a later batch is missing from earlier parent objects. No source documents are stored.


In [ ]:
city = schema["city"]
schema.update([{"city": "Brisbane", "rating": 5, "new_field": "hello", "items": []}])
assert schema.document_count == 4
assert city.count == 4
assert schema["new_field"].missing_count == 3
# An empty array adds no opportunities for an element's price field.
assert price.parent_count == 4
pprint(dict(city.types["str"].value_counts))

## Optional: unambiguous paths

For ordinary nested fields, `items.[].price` is convenient. Tuple paths distinguish
literal dots and `[]` keys from structural traversal. This section is useful when
working with unusual field names; it can be skipped on a first read.

In [ ]:
special = Schema.from_documents([{"a.b": 1, "a": {"b": 2}, "[]": 3}], values="full")
assert special[("a.b",)].types["int"].values == frozenset({1})
assert special["a.b"].types["int"].values == frozenset({2})
assert schema[("items", ARRAY_ELEMENT, "price")] is price
for path, field in special.flat_fields.items():
    print(repr(path), field.types["int"].values if "int" in field.types else "object")
    assert special[path] is field

Try adding an empty object, a missing ancestor, or a nested array and predicting
the counts first. For database data, use `executor.run(..., collect_results=True)`
and pass `result.documents` to `Schema.from_documents()`. Retrieve whole document
bodies; a projection changes the schema you observe, and a limit is not a random sample.

Next: [generate workloads](workload_generation.ipynb) from retained values.